In [1]:
from pyscf import gto, scf, cc

d = 100

o2 = '''
O 0.0 0.0 0.0
O 0.0 0.0 1.20577
'''

m_list = [1]
for nc in m_list:
    atoms = ""
    for n in range(nc):
        shift = n*d
        atoms += f'O {0.0+shift} 0.0 0.0     \n'
        atoms += f'O {0.0+shift} 0.0 1.20577 \n'

    # nfrozen = 2*nc
    spin = 2*nc
    mol = gto.M(atom=atoms, basis="sto6g", spin=spin, verbose=4)
    mol.build()

    mf = scf.UHF(mol)#.density_fit()
    mf.kernel()
    
    stable = False
    while not stable:
        print(f'mean-field stability test')
        if not stable:
            mo_i, _, stable,_ = mf.stability(return_status=True)
            dm = mf.make_rdm1(mo_i,mf.mo_occ)
            mf.kernel(dm0=dm)
        elif stable:
            print(f'UHF Energy: {mf.e_tot}, stability {stable}')
            break


    mycc = cc.CCSD(mf)
    mycc.set_frozen()
    mycc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Wed Jun 10 15:06:42 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 16
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [2]:
import time
import numpy as np
# from jax import random
from jax import numpy as jnp

from afqmc import config
from afqmc import prep, sampling

from functools import partial
print = partial(print, flush=True)
config.setup_jax()

Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      6.17.0-35-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu


In [3]:
from afqmc import integral
integral.prep_integral(mycc, chol_cut=1e-5)


Preparing AFQMC calculation
Calculating Cholesky integrals
Alpha Cholesky shape: (40, 8, 8) 
 Beta Cholesky shape: (40, 8, 8) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [7 5]
Number of basis functions:  8
Number of Cholesky vectors: 40


In [3]:
options = {'eql_time': 10,
           'n_blocks': 100,
           'n_walkers': 100,
           'mix_precision': False,
           'seed': 17,
           'guide': 'uhf',
           'trial': 'uhf',
           }

In [4]:
init_time = time.time()

prep.print_start()

ham, prop, wave, ham_data, wave_data, sampler, options = (prep.init_afqmc_exp(options=options))
ham_data = wave.build_trial_intermediate(ham_data, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, wave, wave_data)
prop_data = prep.init_hf_prop_data(wave, wave_data, ham_data, options)
guide_olps = wave.calc_overlap(prop_data["walkers"], wave_data)
trial_olps = wave.calc_trial_overlap(prop_data["walkers"], wave_data)
wt_init = jnp.sum(prop_data["weights"])
wp_init = jnp.sum(prop_data["weights"] * trial_olps / guide_olps)
e_init = prop_data["e_estimate"]

print("\nEquilibration")
print(f"{'1/T':>5s}  {'weight':>10s}  {'weightp':>10s}  "
      f"{'energy':>12s}  {'realTime':>10s}")
print(f"{0.:5.2f}  {wt_init:10.5f}  {wp_init.real:10.5f}  "
      f"{e_init:12.5f}  {time.time() - init_time:10.2f}")

block_time = prop.dt * sampler.n_prop_steps
neql_block = int(-(-options["eql_time"] // block_time))

for n in range(1,neql_block+1):
    prop_data, (wt, wp, e) \
        = sampler.sample_energy(prop, wave, prop_data, ham_data, wave_data)
    prop_data["e_estimate"] = 0.9 * prop_data["e_estimate"] + 0.1 * e.real

    if (n+1) % (min(max(neql_block // 10, 1), 20)) == 0 and n > 0:
        print(f"{(n+1)*block_time:5.2f}  "
              f"{wt:10.5f}  {wp.real:10.5f}  {e.real:12.5f}  "
              f"{time.time() - init_time:10.2f}")


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               


Load system from Integral File
Maximum memory per walker:            20.00 MB
Maximum number of Cholesky per chunk: 10240
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         40
Number of padding Cholesky:           0

QMC System
Number of electrons: (7, 5)
Spin Multiplicity:   2
Number of orbitals:  8
Number of Chol:    

In [7]:
from afqmc import prep, slater_tools
wave_data = prep.load_cc_amplitude(wave_data)
(wave_data['mo_ta'], wave_data['mo_tb']) = slater_tools.thouless(
                wave_data['mo_coeff'], (wave_data["t1a"], wave_data["t1b"]))

In [9]:
from afqmc.wavefunctions import upt2ccsd_wfn
walker = wave_data["mo_coeff"]
what = upt2ccsd_wfn.calc_energy(wave,walker,ham_data,wave_data)
print(what)
h0 = ham_data["h0"]
energy = (h0 + what[2]/what[0] + what[3]/what[0] - (what[1]*what[2])/(what[0]**2))
print(energy)

[  1.        +0.j   0.        +0.j -46.70911745+0.j  -0.10291952+0.j]
(-149.1610542107657+0j)


In [17]:
def walker_energy(walker, ham_data, wave_data):
    return upt2ccsd_wfn.calc_energy(wave, walker, ham_data, wave_data)

In [18]:
from afqmc import walker_tools
walkers = walker_tools.replicate_walker(walker, 100)
whats = walker_tools.map_over_walkers(walker_energy, walkers, wave.nwalker_batch, ham_data, wave_data)

In [43]:
def weighted_average(weights, samples):
    # weights: (nsample,)
    # samples: (nsample, nterm)
    nsample = len(weights)
    samples = samples.reshape(nsample, -1) # for when only one terms in the samples
    weight_mean = jnp.mean(weights)
    sample_mean = jnp.mean(weights[:, None] * samples, axis=0) / weight_mean

    # weighted variance per term
    deviations = samples - sample_mean
    sample_var = jnp.mean(weights[:, None] * jnp.abs(deviations)**2, axis=0) / weight_mean

    # 1sigma uncertainty of each mean
    sample_err = jnp.sqrt(sample_var / nsample)

    return weight_mean, sample_mean, sample_err

In [ ]:
def pt2ccsd_energy_formula(weights, energy_terms, ham_data):
    # energy_terms shape: (nsamples, terms)
    nsamples = len(weights)
    h0 = ham_data["h0"]
    t1s = energy_terms[:,0]
    t2s = energy_terms[:,1]
    e0s = energy_terms[:,2]
    e1s = energy_terms[:,3]

    weight_mean = jnp.mean(weights)
    t1_mean = jnp.mean(weights * t1s) / weight_mean
    t2_mean = jnp.mean(weights * t2s) / weight_mean
    e0_mean = jnp.mean(weights * e0s) / weight_mean
    e1_mean = jnp.mean(weights * e1s) / weight_mean
    
    energy_mean = (
        h0 + e0_mean/t1_mean + e1_mean/t1_mean - (t2_mean*e0_mean)/t1_mean**2
        ).real

    dE = jnp.array([
         -(e0_mean+e1_mean)/t1_mean**2 + 2*(t2_mean*e0_mean)/t1_mean**3,
         -e0_mean/t1_mean**2,
         1/t1_mean - t2_mean/t1_mean**2,
         1/t1_mean
         ])
    cov_te0e1 = jnp.cov(jnp.stack([t1s, t2s, e0s, e1s]))
    mean_err = (jnp.sqrt(dE @ cov_te0e1 @ dE) / jnp.sqrt((nsamples))).real
    
    terms_mean = jnp.array((t1_mean, t2_mean, e0_mean, e1_mean))

    return weight_mean, terms_mean, (energy_mean, mean_err)


In [39]:
print(jnp.vstack((jnp.array([0,1,2,3,4]),
                 jnp.array([5,6,7,8,9]))).shape)

(2, 5)


In [32]:
weights = jnp.array([1]*100)
weight_mean, compressed_terms, (energy_mean, mean_err) = pt2ccsd_energy_formula(weights, whats, ham_data)
print(weight_mean, energy_mean, mean_err)
print(compressed_terms)

1.0 -149.1610542107657 1.3947701538996772e-18
[  1.        +0.j   0.        +0.j -46.70911745+0.j  -0.10291952+0.j]


In [44]:
weight_mean, sample_mean, sample_err = weighted_average(weights, whats[:,0])
print(weight_mean, sample_mean, sample_err)

1.0 [1.+0.j] [0.]


In [42]:
sample_mean = jnp.mean(weights[:, None] * whats, axis=0) / weight_mean
print(sample_mean)

[  1.        +0.j   0.        +0.j -46.70911745+0.j  -0.10291952+0.j]
